# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neha-raniii/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/neha-raniii/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows loaded")

30,000 rows loaded


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane: Refresh / Content Opportunity Scoring

ML task type: Scoring (a form of ranking). The goal is not to sort content into fixed categories (classification) or find natural groups (clustering) - it's to produce a continuous priority score per page so a reviewer can work down a ranked list, starting with the page most worth their limited time. This is the same shape as the starter pipeline's baseline_refresh_score and final_refresh_score - a number per row, sorted, acted on top-down.

In [15]:
# Task type is conceptual; the dataframe evidence follows in Section 4.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/proxy: is_declining_label = (trend_direction == "down"). This label comes from a defined rule applied to an observed outcome (the current trend bucket), not from a future measured event - which is exactly the beginner-proxy weakness the repo's own guide flags. A stronger version for the capstone would use a genuinely future-looking label: features from a prior window predicting decline over the next window (e.g. prior 90 days -> next 30 days decline), avoiding any overlap between the feature window and the target window.

In [16]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(df['is_declining_label'].value_counts())
print(f"\nDeclining rate: {df['is_declining_label'].mean()*100:.1f}%")


is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining rate: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@50. Because the output is a ranked review queue and a human reviewer only has capacity to check a limited number of pages, the metric that matches the real decision is "of the top 50 pages the system flags first, how many are actually declining?" - not overall accuracy, which would be misleading here since the classes are nearly balanced (54.2% vs 45.8%) but the real use case only cares about the top of the list. This is the same metric the starter pipeline uses (baseline 0.240, random forest 0.740), so it's directly comparable to the reference result.

In [17]:
# Precision@50 will be computed once a model exists (Week 5+); this section defines the metric choice.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one content page (content_id), belonging to one client (client_id). Shown below as an actual dataframe slice with the columns most relevant to this lane: visibility, staleness-adjacent signals, and the current trend label.

In [18]:
lane_slice = df[['content_id', 'client_id', 'impressions_90d', 'days_since_last_update',
                  'avg_position', 'ctr', 'word_count', 'trend_direction', 'is_declining_label']]
print(f"{len(lane_slice):,} rows, one row = one content page")
lane_slice.head(10)


30,000 rows, one row = one content page


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,3080.0,down,1
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,NaN,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,0.09,3807.0,down,1
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,0.16,NaN,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [19]:
"""Why ML beats a fixed rule here: A hand-written rule like "flag if stale AND visible" treats every signal as equally important and combines them with hard thresholds - but Section 1 of my Week-4 baseline work already showed staleness alone gave the OPPOSITE of the expected result on this data (stale pages actually declined less often, not more). A fixed rule can't discover which combination of signals - impressions, position, CTR, word count, staleness - actually matters, or how they interact, or what weight each deserves. A learned model can find that pattern from the data itself and weigh multiple weak signals together, which is exactly why the starter pipeline shows a random forest (Precision@50 = 0.740) beating the hand-written baseline (Precision@50 = 0.240) by roughly 3x - the same evidence I'll be trying to reproduce and extend on my own lane's features."""


'Why ML beats a fixed rule here: A hand-written rule like "flag if stale AND visible" treats every signal as equally important and combines them with hard thresholds - but Section 1 of my Week-4 baseline work already showed staleness alone gave the OPPOSITE of the expected result on this data (stale pages actually declined less often, not more). A fixed rule can\'t discover which combination of signals - impressions, position, CTR, word count, staleness - actually matters, or how they interact, or what weight each deserves. A learned model can find that pattern from the data itself and weigh multiple weak signals together, which is exactly why the starter pipeline shows a random forest (Precision@50 = 0.740) beating the hand-written baseline (Precision@50 = 0.240) by roughly 3x - the same evidence I\'ll be trying to reproduce and extend on my own lane\'s features.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.